## Подготовка среды


In [14]:
%pip -q install -U requests pwntools

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import sys, requests
print('Python:', sys.version.split()[0]); print('requests:', requests.__version__)
try:
    import pwnlib
    print('pwntools: импортирован успешно')
except ImportError as e:
    print('Не удалось импортировать pwntools:', e)

Python: 3.12.4
requests: 2.34.2
pwntools: импортирован успешно


---
## 1. HTTP-запросы с `requests`

HTTP-запрос состоит из метода, URL, заголовков и, иногда, тела. Ответ — статус, заголовки и тело. `requests` понимает HTTP и автоматически формирует запросы.

| Метод/аргумент | Назначение | Пример |
|---|---|---|
| `requests.get(url)` | Получить ресурс | `get(url, timeout=3)` |
| `requests.post(url)` | Отправить данные | `post(url, json={'x': 1})` |
| `params={...}` | Параметры URL | `?code=042` |
| `data={...}` | Тело HTML-формы | `data={'name': 'alice'}` |
| `json={...}` | JSON-тело; автоматически задаёт Content-Type | `json={'x': 2}` |
| `headers={...}` | Заголовки клиента | `headers={'User-Agent':'CTF-Lesson'}` |
| `cookies={...}` | Cookies запроса | `cookies={'mode':'training'}` |
| `auth=(user, password)` | HTTP Basic Auth | `auth=('u', 'p')` |
| `timeout=3` | Таймаут; `(connect, read)` | `timeout=(2,3)` |
| `allow_redirects=False` | Не следовать редиректам | `get(url, allow_redirects=False)` |
| `verify=True` | Проверять TLS-сертификат (не отключать) | применяется для HTTPS |
| `files={...}` | Multipart-загрузка файла | `files={'file': f}` |
| `requests.Session()` | Сохранять cookies и переиспользовать соединения | `with Session() as s:` |

Ответы:

| Поле ответа | Смысл |
|---|---|
| `.status_code`, `.ok` | HTTP-статус и успешность (`<400`) |
| `.text`, `.content` | Текст или сырые `bytes` |
| `.json()` | Декодирование JSON (может вызвать исключение) |
| `.headers`, `.cookies`, `.url` | Заголовки, cookies и фактический URL |
| `.history` | История редиректов |
| `.raise_for_status()` | Исключение для кодов 4xx/5xx |

Статусы, которые полезно знать: **200**, **201**, **301/302**, **400**, **401**, **403**, **404**, **429**, **500**.

### Запускаем учебный HTTP-сервис


In [16]:
import json
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs

class CTFHTTPHandler(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass

    def respond(self, status, obj, *, cookie=None):
        data = json.dumps(obj, ensure_ascii=False).encode('utf-8')
        self.send_response(status)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if cookie: self.send_header('Set-Cookie', cookie)
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        parsed = urlparse(self.path)
        q = parse_qs(parsed.query)
        if parsed.path == '/api/info':
            return self.respond(200, {'result': {'flag': 'CTF{http_json_ok}'}, 'message': 'Прочитай JSON'})
        if parsed.path == '/api/check':
            guess = q.get('code', [''])[0]
            return self.respond(200 if guess == '042' else 403,
                                {'ok': guess == '042', 'flag': 'CTF{http_code_found}' if guess == '042' else None})
        if parsed.path == '/api/cookie':
            if 'training=yes' in self.headers.get('Cookie', ''):
                return self.respond(200, {'flag': 'CTF{cookie_kept}'})
            return self.respond(200, {'step': 'cookie_set'}, cookie='training=yes; Path=/')
        return self.respond(404, {'error': 'not found'})

    def do_POST(self):
        length = int(self.headers.get('Content-Length', '0'))
        body = self.rfile.read(length)
        ctype = self.headers.get('Content-Type', '')
        if self.path == '/api/echo':
            if 'application/json' in ctype:
                try: parsed = json.loads(body)
                except json.JSONDecodeError: return self.respond(400, {'error': 'invalid JSON'})
            else:
                parsed = {k: v[0] for k, v in parse_qs(body.decode()).items()}
            return self.respond(200, {'received': parsed, 'content_type': ctype})
        return self.respond(404, {'error': 'not found'})

http_server = ThreadingHTTPServer(('127.0.0.1', 0), CTFHTTPHandler)
http_port = http_server.server_address[1]
threading.Thread(target=http_server.serve_forever, daemon=True).start()
HTTP_BASE = f'http://127.0.0.1:{http_port}'
print('HTTP-сервис:', HTTP_BASE)

HTTP-сервис: http://127.0.0.1:1583


### Демонстрация 1. Простой GET: статус, текст, JSON

`GET` запрашивает данные. `r.text` — текст ответа, `r.content` — байты, `r.json()` — Python-словарь (если ответ содержит JSON). Статус `200` обычно означает успех. JSON может содержать вложенные словари.

In [17]:
r = requests.get(HTTP_BASE + '/api/info', timeout=3)
print('тип ответа:', type(r).__name__)
print('HTTP:', r.status_code)
print('тип данных JSON:', type(r.json()).__name__)
print('поля верхнего уровня:', list(r.json().keys()))
print('сообщение:', r.json()['message'])

тип ответа: Response
HTTP: 200
тип данных JSON: dict
поля верхнего уровня: ['result', 'message']
сообщение: Прочитай JSON


### Демонстрация 2. URL-параметры и статусы

В `params={'code': '042'}` **значение строки** `'042'` важно: число `42` потеряло бы ведущий ноль. `requests` кодирует параметры URL самостоятельно.

Проверим правильный и неправильный код: при неверном коде учебный сервер отвечает `403`. Это **не исключение** само по себе; исключение появляется после `raise_for_status()`.

In [ ]:
for guess in ['001', '042']:
    r = requests.get(HTTP_BASE + '/api/check', params={'code': guess}, timeout=3)
    print('URL:', r.url)
    print('код:', guess, 'HTTP:', r.status_code, 'ответ:', r.json())

### Демонстрация 3. Заголовки и cookies

`headers` — метаданные запроса. Cookies хранят состояние между обращениями к серверу. `Session()` автоматически сохраняет полученные cookies: сервер сначала выдаёт cookie, затем клиент отправляет её в следующем запросе.

In [ ]:
with requests.Session() as session:
    first = session.get(HTTP_BASE + '/api/cookie', headers={'User-Agent': 'CTF-Class/1.0'}, timeout=3)
    print('первый ответ:', first.json())
    print('cookies в сессии:', session.cookies.get_dict())
    second = session.get(HTTP_BASE + '/api/cookie', timeout=3)
    print('второй ответ:', second.json())

In [ ]:
# GET, статус, JSON и парсинг вложенного значения
r = requests.get(HTTP_BASE + '/api/info', timeout=3)
print('status:', r.status_code)
print('text:', r.text)
print('bytes:', r.content)
print('flag:', r.json()['result']['flag'])
r.raise_for_status()

### GET vs POST; `data` vs `json`

В `requests.post(url, data={'x': 1})` тело обычно формируется как HTML-форма (`application/x-www-form-urlencoded`), а `json={'x': 1}` создаёт JSON-тело (`application/json`). С серверной стороны способ разбора отличается. Напечатайте ответы и найдите отличие `content_type`.

In [ ]:
# POST-форма и POST-JSON
form = requests.post(HTTP_BASE + '/api/echo', data={'name': 'Alice', 'score': '7'}, timeout=3)
js = requests.post(HTTP_BASE + '/api/echo', json={'name': 'Alice', 'score': 7}, timeout=3)
print('data= →', form.json())
print('json= →', js.json())

# Session сохраняет cookies между запросами
with requests.Session() as session:
    print('Первый ответ:', session.get(HTTP_BASE + '/api/cookie', timeout=3).json())
    print('Второй ответ:', session.get(HTTP_BASE + '/api/cookie', timeout=3).json())

### Перед перебором: цикл, условие, `break`

`range(100)` даёт числа от 0 до 99. `f'{number:03d}'` превращает `7` в `'007'`. При каждом запросе проверяем поле `ok` в JSON. `break` остановит цикл после успеха, чтобы не посылать лишние запросы. Перебираем **только локальный учебный сервис**.

In [ ]:
for number in range(4):
    candidate = f'{number:03d}'
    print(number, '->', candidate)

example = {'ok': False, 'flag': None}
if example['ok']:
    print('нашли')
else:
    print('ещё не нашли')

In [ ]:
# Мини-задача: всего 100 комбинаций (000–099)
for number in range(100):
    guess = f'{number:03d}'
    r = requests.get(HTTP_BASE + '/api/check', params={'code': guess},
                     headers={'User-Agent': 'CTF-Lesson'}, timeout=3)
    if r.status_code == 200:
        print('Успех:', guess, r.json()['flag'], 'URL:', r.url)
        break

**Практика 1:** получить `/api/info`, вывести только флаг

---
## 2. TCP и `socket`

`requests` работает с HTTP, а `socket` — напрямую с транспортным соединением. TCP — **поток байтов**: `recv(1024)` может вернуть меньше 1024 байт или лишь часть строки.

| API / аргумент | Что делает |
|---|---|
| `socket.socket(AF_INET, SOCK_STREAM)` | IPv4 + TCP (`SOCK_DGRAM` — UDP) |
| `socket.create_connection((host, port), timeout=3)` | Установить TCP-соединение |
| `s.connect((host,port))` | Подключиться к адресу |
| `s.settimeout(3)` | Таймаут последующих операций |
| `s.send(bytes)` | Может отправить только часть сообщения |
| `s.sendall(bytes)` | Отправить все байты или вызвать ошибку |
| `s.recv(1024)` | Получить **не больше** 1024 байт |
| `s.makefile('rwb')` | Файловый интерфейс; есть `readline()` |
| `s.shutdown(socket.SHUT_WR)` | Закрыть направление отправки |
| `s.close()` | Закрыть соединение (обычно через `with`) |

Для парсинга: `.decode()`, `.encode()`, `.strip()`, `.split()`, `int()`, `re`.

### Локальный TCP-сервис

Сервис отправляет строку `Solve: 19 + 23 = ?`, ожидает ответ и возвращает флаг.

### Как `socket` видит сеть

TCP — поток байтов, а не отдельные готовые сообщения. `socket.create_connection((host, port), timeout=3)` соединяется с сервером. Обратите внимание на **двойные скобки**: `(host, port)` — один аргумент-кортеж, `timeout=3` — второй именованный аргумент.

В полученном сообщении `b'Hello\n'` буква `b` обозначает bytes, а `\n` — перевод строки. `sendall()` отправляет bytes; `.encode()` делает их из строки. `recv(1024)` может вернуть только **часть** данных, поэтому для текстового протокола с `\n` удобен `makefile().readline()`.

In [ ]:
import socket
print('TCP-константа SOCK_STREAM:', socket.SOCK_STREAM)
text = 'PING\n'
print('str:', repr(text))
print('bytes:', repr(text.encode()))
print('длина в байтах:', len(text.encode()))

In [ ]:
import socket
import socketserver

class MathHandler(socketserver.StreamRequestHandler):
    def handle(self):
        self.wfile.write(b'Solve: 19 + 23 = ?\n')
        self.wfile.flush()
        answer = self.rfile.readline().strip()
        self.wfile.write(b'CTF{socket_math_ok}\n' if answer == b'42' else b'Wrong answer\n')
        self.wfile.flush()

math_server = socketserver.ThreadingTCPServer(('127.0.0.1', 0), MathHandler)
math_server.daemon_threads = True
math_port = math_server.server_address[1]
threading.Thread(target=math_server.serve_forever, daemon=True).start()
print('TCP-сервис:', '127.0.0.1', math_port)

### Демонстрация: получить приветствие и отправить ответ

Сначала сделаем всё вручную, без регулярных выражений. `with` закроет сокет; `readline()` читает до `\n`, а `stream.write()` пишет байты в буфер. После `write()` вызываем `flush()` — иначе сервер может ждать данные, которые остаются в буфере клиента.

In [ ]:
with socket.create_connection(('127.0.0.1', math_port), timeout=3) as sock:
    stream = sock.makefile('rwb')
    prompt = stream.readline()
    print('сырые байты:', prompt)
    print('текст:', prompt.decode().strip())
    stream.write(b'42\n')
    stream.flush()
    print('ответ сервера:', stream.readline().decode().strip())

### Как работает парсинг ответа

`re.search(r'(\d+)\s*\+\s*(\d+)', text)` ищет два числа через `+` с необязательными пробелами. `r'...'` — raw-строка, в которой обратный слеш удобно использовать для регулярных выражений; `\d+` означает одну или более цифр, `\s*` — ноль или больше пробельных символов. `m.group(1)` и `m.group(2)` возвращают части, заключённые в скобки шаблона.

In [ ]:
import re
sample = 'Solve: 19 + 23 = ?'
m = re.search(r'(\d+)\s*\+\s*(\d+)', sample)
print('совпадение:', m.group(0))
print('первое число:', m.group(1), 'второе:', m.group(2))
print('сумма:', int(m.group(1)) + int(m.group(2)))

In [ ]:
import re
with socket.create_connection(('127.0.0.1', math_port), timeout=3) as s:
    stream = s.makefile('rwb')
    task = stream.readline().decode('utf-8').strip()
    print('Сервер:', task)
    x, y = map(int, re.findall(r'\d+', task))
    result = x + y
    stream.write(f'{result}\n'.encode())
    stream.flush()
    print('Ответ сервера:', stream.readline().decode().strip())

**Практика 2:** поменяйте в сервере сложение на умножение и адаптируйте клиент. Почему `sendall(b"42")` может не сработать, а `sendall(b"42\n")` — сработает? (`readline()` ждёт `\n`.)

---
## 3. Перебор вариантов и `itertools`

| Функция | Учитывает порядок | Повторения | Количество (N символов, длина k) |
|---|---|---|---|
| `product(alphabet, repeat=k)` | Да | Да | `N**k` |
| `permutations(alphabet, r=k)` | Да | Нет¹ | `N!/(N-k)!` |
| `combinations(alphabet, r=k)` | Нет | Нет¹ | `N!/(k!(N-k)!)` |
| `combinations_with_replacement(alphabet, r=k)` | Нет | Да | `(N+k-1)!/(k!(N-1)!)` |

¹ Элементы считаются разными по **позиции** во входном наборе; при дубликатах во входе видимые значения могут повторяться.

`product` принимает и разные наборы, например `product('AB', '12')`. Для огромного числа вариантов используйте **итератор**: не оборачивайте `product(...)` в `list(...)` без необходимости.

### Что такое `itertools`: не готовые пароли, а последовательности элементов

`product('AB', repeat=2)` последовательно выдаёт кортежи `('A', 'A')`, `('A', 'B')` и т. д. Это **итератор**: в обычном переборе он выдаёт следующий вариант по мере необходимости. `''.join(chars)` превращает кортеж символов в строку.

Различайте:
- `product` — порядок важен, повторения разрешены;
- `permutations` — порядок важен, одна и та же позиция входа не используется дважды;
- `combinations` — порядок выбора не важен;
- `combinations_with_replacement` — сочетания с повторениями.

В таблице ниже `N` — размер входного набора, `k` — длина результата.

In [ ]:
from itertools import product, permutations, combinations
print('product:')
for chars in product('AB', repeat=2):
    print(chars, '=>', ''.join(chars))
print('permutations:', [''.join(x) for x in permutations('ABC', 2)])
print('combinations:', [''.join(x) for x in combinations('ABC', 2)])
print('разные наборы:', list(product('AB', '12')))

### Как оценить число попыток

У `product` при алфавите из `N` символов и длине `k` будет `N ** k` вариантов. Например, `3 ** 4 = 81`. `list(product(...))` годится для маленьких примеров, но для большого перебора лучше итерироваться непосредственно в `for`, не сохраняя все варианты в памяти.

In [ ]:
alphabet = 'ABC'
for length in [1, 2, 3, 4]:
    print('длина', length, 'вариантов:', len(alphabet) ** length)
print('первые 5:', list(__import__('itertools').islice(product(alphabet, repeat=4), 5)))

In [ ]:
from itertools import product, permutations, combinations, combinations_with_replacement
print('product:', list(product('AB', repeat=2)))
print('permutations:', list(permutations('ABC', 2)))
print('combinations:', list(combinations('ABC', 2)))
print('combinations_with_replacement:', list(combinations_with_replacement('AB', 2)))
print('разные наборы:', list(product('AB', '12')))
print('варианты из 10 цифр длиной 4:', 10**4)

### Хеширование: почему сравниваем строки хешей

`hashlib.sha256(data)` принимает **bytes**, а не `str`. `.hexdigest()` возвращает шестнадцатеричную строку фиксированной длины. Для учебного перебора считаем SHA-256 от каждого кандидата и сравниваем с заранее известным хешем. Это не «расшифровка» хеша, а проверка гипотез перебором.

In [ ]:
import hashlib
for word in ['CAT', 'DOG']:
    digest = hashlib.sha256(word.encode()).hexdigest()
    print(word, '=>', digest[:16] + '...')
print('Проверка:', hashlib.sha256(b'CAT').hexdigest() == hashlib.sha256('CAT'.encode()).hexdigest())

In [ ]:
# Учебный перебор SHA-256 от трёхсимвольного кода; не атакуем реальные аккаунты
import hashlib
alphabet = 'AB2'
secret_for_teacher = 'B2A'
target_digest = hashlib.sha256(secret_for_teacher.encode()).hexdigest()
print('Известный SHA-256:', target_digest)
for chars in product(alphabet, repeat=3):
    guess = ''.join(chars)
    if hashlib.sha256(guess.encode()).hexdigest() == target_digest:
        print('Найдено:', guess)
        break

**Практика 3:** не глядя на `secret_for_teacher`, придумайте перебор строк длиной от 1 до 3. Используйте вложенные циклы `for length in range(1, 4)` и `product(..., repeat=length)`.

---
## 4. CLI: `argparse`, `os`, `subprocess`, `sys`

Скрипты CTF удобнее запускать с параметрами (`--host`, `--port`, `-n`), чем изменять код каждый раз.

| Модуль/опция | Значение |
|---|---|
| `argparse.ArgumentParser(description=...)` | Описание CLI |
| `.add_argument('message')` | Позиционный аргумент |
| `.add_argument('-n','--repeat', type=int, default=1)` | Короткая/длинная опция, тип, умолчание |
| `required=True` | Опция обязательна |
| `choices=['http','tcp']` | Допустимые значения |
| `action='store_true'` | Флаг без значения, включается наличием |
| `help='...'`; `-h`, `--help` | Подсказки |
| `sys.argv`, `sys.executable`, `sys.exit(code)` | Аргументы, интерпретатор, код выхода |
| `os.getcwd()`, `os.listdir('.')`, `os.environ.get('NAME')` | Каталог и переменные окружения |
| `subprocess.run([...], capture_output=True)` | Запустить команду, захватить вывод |
| `text=True`, `check=True`, `timeout=5` | Декодирование, проверка кода, ограничение времени |
| `cwd='...'`, `env={...}` | Рабочий каталог и окружение подпроцесса |

**Важно для Colab:** при обычном `parser.parse_args()` читаются аргументы процесса Jupyter. Вместо этого передаём **список параметров вручную**: `parser.parse_args(['--repeat', '3', ...])`. В обычном `.py` вы бы просто вызвали `parse_args()`.

### Синтаксис CLI: позиционные аргументы и флаги

В обычном терминале команда `python tool.py hello -n 3 --uppercase` содержит:

- `hello` — **позиционный** аргумент;
- `-n 3` / `--repeat 3` — опция со значением (`int`);
- `--uppercase` — **флаг-переключатель** без значения (`action='store_true'`);
- `--mode brackets` — ограниченный выбор (`choices=[...]`).

`default=1` используется, если параметр не указан, `required=True` делает именованную опцию обязательной. `parse_args([...])` удобно для Colab: мы вручную передаём список так, словно это аргументы командной строки. Реальный запуск из терминала использовал бы `parse_args()` без списка.

In [ ]:
import argparse
parser = argparse.ArgumentParser(description='Первый CLI')
parser.add_argument('name', help='Кого приветствовать')
parser.add_argument('-n', '--count', type=int, default=1)
parser.add_argument('--loud', action='store_true')
for argv in [['Alice'], ['Bob', '-n', '2', '--loud']]:
    args = parser.parse_args(argv)
    print('вход:', argv, '=>', vars(args))
    for _ in range(args.count):
        print(('hello ' + args.name).upper() if args.loud else 'hello ' + args.name)

In [ ]:
import argparse, os, subprocess, sys

def cli_demo(argv):
    parser = argparse.ArgumentParser(description='Учебный CLI для CTF')
    parser.add_argument('message', help='Текст')
    parser.add_argument('-n', '--repeat', type=int, default=1, help='Число повторов')
    parser.add_argument('--uppercase', action='store_true', help='Переводить в верхний регистр')
    parser.add_argument('--mode', choices=['plain', 'brackets'], default='plain')
    args = parser.parse_args(argv)
    result = args.message.upper() if args.uppercase else args.message
    if args.mode == 'brackets': result = f'[{result}]'
    for _ in range(args.repeat): print(result)

cli_demo(['hello', '-n', '3', '--uppercase', '--mode', 'brackets'])
print('--- Доступные опции ---')
try:
    cli_demo(['--help'])
except SystemExit as exc:
    print('argparse завершился с кодом:', exc.code)

### `os`, `sys`, `subprocess`: разные задачи

- `os.getcwd()` возвращает текущую директорию, `os.environ.get('KEY', default)` читает переменную окружения.
- `sys.executable` — путь к Python, который выполняет блокнот, `sys.argv` — аргументы **текущего процесса Colab**, поэтому они не похожи на аргументы CLI-скрипта.
- `subprocess.run([program, arg1, ...], ...)` запускает дочерний процесс. **Список аргументов безопаснее**, чем собирать команду в строку с `shell=True`.
- `capture_output=True` сохраняет stdout/stderr; `text=True` преобразует байты в строки; `check=True` вызывает исключение при ненулевом коде завершения; `timeout=5` ограничивает ожидание процесса.

In [ ]:
import os, sys, subprocess
print('рабочая папка:', os.getcwd())
print('исполняемый Python:', sys.executable)
print('переменная HOME:', os.environ.get('HOME', '<не установлена>'))
result = subprocess.run([sys.executable, '-c', 'print(6 * 7)'], capture_output=True, text=True, timeout=5, check=True)
print('stdout:', result.stdout.strip(), 'код возврата:', result.returncode)

In [ ]:
# Данные среды и безопасный запуск дочернего Python-процесса
print('Папка:', os.getcwd())
print('Python:', sys.executable)
print('CTF_MODE:', os.environ.get('CTF_MODE', 'не задан'))
res = subprocess.run([sys.executable, '-c', 'print(6 * 7)'],
                     capture_output=True, text=True, timeout=5, check=True)
print('stdout:', res.stdout.strip(), '| stderr:', repr(res.stderr), '| code:', res.returncode)

**Практика 4:** добавьте флаг `--host` и порт `--port` в парсер. Проверку диапазона `1..65535` сделайте с помощью `parser.error('...')`.

---
## 5. `pwntools`

`pwntools` удобна для взаимодействия с сервисами, работы с байтовыми потоками и построения payload.

| API | Назначение |
|---|---|
| `remote(host, port, timeout=3)` | TCP-подключение |
| `process(['./binary'])` | Запуск локального исполняемого файла (Linux) |
| `io.recv(n, timeout=...)` | Принять до `n` байт |
| `io.recvline(timeout=...)` | Принять строку до `\n` |
| `io.recvuntil(b'> ', drop=False)` | Принять до маркера; `drop=True` убирает его |
| `io.send(b'data')` | Отправить байты |
| `io.sendline(b'data')` | Отправить байты и `\n` |
| `io.sendafter(prompt, data)` | Принять prompt, затем `send()` |
| `io.sendlineafter(prompt, data)` | Принять prompt, затем `sendline()` |
| `io.interactive()` | Перейти к интерактивному вводу (в Colab неудобно) |
| `context.log_level='debug'` | Детальный лог получения и отправки |
| `context.arch='amd64'`, `context.endian='little'` | Архитектура и порядок байтов |

**Упаковка целых чисел:** `p32(x)`, `p64(x)` возвращают 4/8 байт в заданном порядке, а `u32(bytes)`, `u64(bytes)` — числа. `flat(...)` объединяет поля в payload. `cyclic(size, n=4)` и `cyclic_find(fragment, n=4)` помогают определять смещения. При изменении `n` используйте его в обеих функциях.

Здесь мы используем **синтетическую утечку адреса**, не настоящую уязвимость.

### Зачем `pwntools`, если есть `socket`?

`pwntools` специально упрощает взаимодействие с CTF-сервисами: `remote()` устанавливает соединение, `recvuntil(b'> ')` получает данные до приглашения, `sendline()` добавляет `\n`, а `sendlineafter()` выполняет два действия подряд. Это сокращает типичный CTF-скрипт.

**Payload** — последовательность байтов, которую отправляем программе. Адреса и числа в памяти часто лежат в виде *little-endian*: младший байт идёт первым. `p32()`/`p64()` упаковывают число в 4/8 байт, `u32()`/`u64()` выполняют обратную операцию.

`flat()` удобно объединяет куски payload; `cyclic()` создаёт узнаваемый шаблон для определения смещения. Ниже сначала разберём упаковку без подключения к серверу.

In [ ]:
from pwn import p32, p64, u32, u64, flat, cyclic, cyclic_find, context
context.arch = 'amd64'
context.endian = 'little'
value = 0x12345678
packed = p32(value)
print('число:', hex(value))
print('байты:', packed, 'hex:', packed.hex())
print('обратно:', hex(u32(packed)))
print('p64 length:', len(p64(value)))
print('payload:', flat(b'ABCD', p64(value), word_size=64).hex())

### Демонстрация: утечка из 6 байт и поиск смещения

На 64-битной архитектуре адрес удобно обрабатывать как 8 байт. Если учебный сервис передал только 6 младших байт адреса, `leak.ljust(8, b'\x00')` добавит справа нулевые байты до длины 8. Это подходит к **нашему синтетическому примеру**, но не является универсальным правилом обработки любых утечек.

`cyclic(64)` создаёт шаблон, в котором короткие фрагменты уникальны в заданном диапазоне. `cyclic_find(pattern[20:24])` ищет позицию этого фрагмента.

In [ ]:
from pwn import cyclic, cyclic_find
leak = bytes.fromhex('90785634127f')
print('6 байт:', leak.hex())
print('добавили нули:', leak.ljust(8, b'\x00').hex())
print('адрес:', hex(u64(leak.ljust(8, b'\x00'))))
pattern = cyclic(64)
print('шаблон:', pattern[:32])
print('фрагмент:', pattern[20:24], 'смещение:', cyclic_find(pattern[20:24]))

In [ ]:
from pwn import remote, p32, p64, u32, u64, flat, cyclic, cyclic_find, context
context.log_level = 'error'  # можно поменять на 'debug' для разбора трафика
context.arch = 'amd64'
context.endian = 'little'
print('p32:', p32(0x12345678).hex())
print('p64:', p64(0x1122334455667788).hex())
print('u64:', hex(u64(bytes.fromhex('8877665544332211'))))
leak = bytes.fromhex('90785634127f')
address = u64(leak.ljust(8, b'\x00'))
print('Из 6 байтов:', hex(address))
print('payload:', flat(b'TAG:', p64(address), word_size=64).hex())
pattern = cyclic(100)
print('cyclic_find:', cyclic_find(pattern[24:28]))

In [ ]:
# Локальный TCP-сервис, который отдает синтетическую утечку
SYNTHETIC_ADDRESS = 0x00007F1234567890

class PwnHandler(socketserver.StreamRequestHandler):
    def handle(self):
        raw = SYNTHETIC_ADDRESS.to_bytes(8, 'little')[:6]
        self.wfile.write(b'LEAK: ' + raw.hex().encode() + b'\n')
        self.wfile.write(b'Payload (hex): ')
        self.wfile.flush()
        received = self.rfile.readline().strip()
        expected = (b'TAG:' + SYNTHETIC_ADDRESS.to_bytes(8, 'little')).hex().encode()
        self.wfile.write(b'CTF{pwntools_packing_ok}\n' if received == expected else b'Incorrect payload\n')
        self.wfile.flush()

pwn_server = socketserver.ThreadingTCPServer(('127.0.0.1', 0), PwnHandler)
pwn_server.daemon_threads = True
pwn_port = pwn_server.server_address[1]
threading.Thread(target=pwn_server.serve_forever, daemon=True).start()
print('Учебный Pwn-сервис:', '127.0.0.1', pwn_port)

### Синтаксис подключения к сервису

`io = remote('127.0.0.1', pwn_port, timeout=3)` подключается к локальному серверу. `try ... finally` гарантирует `io.close()`; `.recvline().strip()` получает и очищает строку; `.split(b': ', 1)[1]` берёт всё после первого разделителя. `bytes.fromhex()` переводит шестнадцатеричное представление в сырые байты. `sendlineafter(b'Payload (hex): ', data)` дожидается приглашения и отправляет строку с переводом строки.

Попробуйте поставить `context.log_level = 'debug'` и сравнить реальные отправленные/полученные байты с тем, что печатает скрипт.

In [ ]:
io = remote('127.0.0.1', pwn_port, timeout=3)
try:
    line = io.recvline().strip()
    print('Принято:', line)
    leak_bytes = bytes.fromhex(line.split(b': ', 1)[1].decode())
    leaked_address = u64(leak_bytes.ljust(8, b'\x00'))
    payload = flat(b'TAG:', p64(leaked_address), word_size=64)
    io.sendlineafter(b'Payload (hex): ', payload.hex().encode())
    print('Результат:', io.recvline().decode().strip())
finally:
    io.close()

**Практика 5:** в ячейке сервера измените `SYNTHETIC_ADDRESS`, повторно запустите сервер и клиент. Клиент должен обработать новую утечку без изменений. Попробуйте поставить `context.log_level = 'debug'` и наблюдать чтение и отправку.

---
## 6. Итоговая задача

Локальный TCP-сервис выдаёт:

```text
ALPHABET=AB2
LENGTH=3
SHA256=<хеш неизвестной строки>
CODE:
```

Нужно распарсить строки, сгенерировать варианты через `itertools.product`, сравнить SHA-256 и отправить найденный код. Сначала предложите ученикам написать **своё решение в следующей пустой ячейке**; затем покажите образец ниже.

**Дополнительная сложность:** использовать `pwntools` вместо `socket` и обернуть решение в функцию с аргументами `host`, `port`.

---
## 6. Итоговая учебная задача

Сервис пришлёт **четыре строки**: алфавит, длину пароля, SHA-256 и приглашение `CODE:`. Требуется распарсить эти данные, перебрать подходящие строки и отправить правильный ответ.

**Алгоритм решения:**
1. Получить строки от сервиса через `socket` или `pwntools`.
2. Разделить строки вида `KEY=VALUE` с помощью `.split('=', 1)`.
3. Получить `alphabet`, `length` и `digest`.
4. Сгенерировать кандидатов функцией `product`.
5. Сравнить `hashlib.sha256(candidate.encode()).hexdigest()` с нужным хешем.
6. Отправить код с `\n` и прочитать флаг.

Это небольшой учебный перебор (27 вариантов), не попытка угадать пароль чужого аккаунта.

In [ ]:
FINAL_SECRET = '2BA'

class FinalHandler(socketserver.StreamRequestHandler):
    def handle(self):
        digest = hashlib.sha256(FINAL_SECRET.encode()).hexdigest().encode()
        for line in [b'ALPHABET=AB2\n', b'LENGTH=3\n', b'SHA256=' + digest + b'\n', b'CODE:\n']:
            self.wfile.write(line)
        self.wfile.flush()
        answer = self.rfile.readline().strip().decode(errors='replace')
        self.wfile.write(b'CTF{python_automation_complete}\n' if answer == FINAL_SECRET else b'Wrong\n')
        self.wfile.flush()

final_server = socketserver.ThreadingTCPServer(('127.0.0.1', 0), FinalHandler)
final_server.daemon_threads = True
final_port = final_server.server_address[1]
threading.Thread(target=final_server.serve_forever, daemon=True).start()
print('Итоговый сервис:', '127.0.0.1', final_port)

In [ ]:
# МЕСТО ДЛЯ САМОСТОЯТЕЛЬНОГО РЕШЕНИЯ
# Используйте ('127.0.0.1', final_port).
# Подсказка: можно подключиться через socket.create_connection(...)

### Образец решения (показывать после самостоятельной работы)

Используем TCP, генератор и SHA-256. Сервер ждёт строку с переводом строки.

In [ ]:
with socket.create_connection(('127.0.0.1', final_port), timeout=3) as sock:
    stream = sock.makefile('rwb')
    lines = [stream.readline().decode().strip() for _ in range(4)]
    parameters = dict(line.split('=', 1) for line in lines if '=' in line)
    alphabet = parameters['ALPHABET']
    length = int(parameters['LENGTH'])
    digest = parameters['SHA256']
    print('Условие:', parameters)
    for chars in product(alphabet, repeat=length):
        candidate = ''.join(chars)
        if hashlib.sha256(candidate.encode()).hexdigest() == digest:
            stream.write((candidate + '\n').encode())
            stream.flush()
            print('Подобран код:', candidate)
            print('Сервис:', stream.readline().decode().strip())
            break

## Дополнительные самостоятельные мини-задания

1. **HTTP:** добавьте `params={'code': '042'}` к запросу и выведите `r.url`. Почему `%20` иногда заменяет пробелы в URL?
2. **HTTP:** сделайте два запроса к `/api/cookie` без сессии, а затем с `Session()`. Почему результаты отличаются?
3. **TCP:** поменяйте в клиенте `b'42\n'` на `b'41\n'`. Как ведёт себя сервер?
4. **Перебор:** выведите все двухсимвольные комбинации из `ABC`, затем только те, где буквы не повторяются.
5. **CLI:** добавьте в `argparse` параметр `--port` с типом `int` и значением по умолчанию `31337`.
6. **Pwntools:** упакуйте `0xdeadbeef` как 32-битное число и восстановите его из байтов.

**Вопросы для проверки понимания:** чем отличается `str` от `bytes`? Почему `recv(1024)` не гарантирует получение целой строки? Чем `product` отличается от `permutations`? Зачем нужны `timeout` и `with`? Что делает `sendlineafter()`?

---
## Справочные ссылки

- [requests: Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/)
- [Python: socket](https://docs.python.org/3/library/socket.html)
- [Python: itertools](https://docs.python.org/3/library/itertools.html)
- [Python: argparse](https://docs.python.org/3/library/argparse.html)
- [Python: subprocess](https://docs.python.org/3/library/subprocess.html)
- [pwntools: tubes](https://docs.pwntools.com/en/stable/tubes.html)